In [ ]:
import pandas as pd
import importlib
import numpy as np

import plotly.express as px
import matplotlib.pyplot as plt

import irina.utility_functions as uf
from irina.cosine_yearly_analysis import df_country_subfield

In [ ]:
def get_stats_year(year, path="df_country_subfield/"):
    df_year = pd.read_csv(uf.PATH+path+f"{year}.csv", index_col="country")
    return (
        uf.get_country_stats(df_year)
        .assign(log_total_articles=lambda df: np.log(df.total_articles))
        .merge(uf.df_country[["alpha-2", "alpha-3", "region"]], left_index=True, right_on="alpha-2", how="left")
    )

def read_country_subfield(year, type_name, df_global):
    return pd.concat([
            pd.read_csv(uf.PATH+f"df_country_subfield/{type_name}_{year}.csv", index_col=0),
            df_global.query("year == @year").assign(subfield=lambda df: df.subfield_id.astype(str)).set_index("subfield").counts_individual.to_frame("Global").T
        ], axis=0)

def get_flat_country_subfield(df_country_subfield):
    return (
        df_country_subfield
        .reset_index(names=["country"])
        .melt(
            id_vars=["country"],
            value_vars=df_country_subfield.columns.to_list(),
            var_name="subfield_id",
            value_name="counts",
        )
    )

def get_top_n_sorted(df, year, type_name, n=10):
    return (
        df
        .sum(axis=1)
        .to_frame("total_production")
        .assign(global_share=lambda df: df.total_production / df.total_production.iloc[-1])
        .sort_values("global_share", ascending=False)
    ).head(n+1).reset_index(names="country")[["country", "global_share"]].assign(year=year, type=type_name).iloc[1:].reset_index(names="position")

In [ ]:
type_list = ["individual", "collaborative", "w_countries", "w_authors"]

In [ ]:
df_global = pd.read_csv(uf.PATH+"df_global_comparison_yearly.csv").drop_duplicates()

# Top 10 in the world

In [ ]:
year = 1990
df_by_types = []
for type_loop in type_list:
    df_country_subfield_tmp = read_country_subfield(year, type_loop, df_global)
    df_by_types.append(get_top_n_sorted(df_country_subfield_tmp, year, type_loop))
df_top_by_type = pd.concat(df_by_types)

In [ ]:

top_n = 10
all_countries = df_top_by_type["country"].unique()

# Assign a consistent color to each country
cmap = plt.cm.tab20.colors
country_colors = {country: cmap[i % len(cmap)] for i, country in enumerate(all_countries)}
others_color = "lightgray"

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(12, 10))

fig.suptitle("Share in global scientific production (individual)", fontsize=16, y=1.02)

for i, ax in enumerate(axes.flatten()):
    # print(ax)
    type_loop = type_list[i]
    df_year = df_top_by_type.query("type == @type_loop").sort_values("global_share", ascending=False)

    # Take top_n countries
    # df_top = df_year.head(top_n)

    values = list(df_year.global_share)
    labels = list(df_year.country)

    # Add "Others" slice
    sum_top = sum(values)
    values.append(1 - sum_top)
    labels.append("Others")

    # Assign colors for this pie
    colors = [country_colors.get(c, others_color) for c in labels]
    colors[-1] = others_color  # make "Others" gray

    # Draw pie chart
    ax.pie(
        values,
        labels=labels,
        autopct="%.1f%%",
        startangle=90,
        pctdistance=0.85,  # move percentages closer to the edge
        colors=colors
    )
    ax.set_title(str(type_loop))

plt.tight_layout()
# plt.show()
# plt.savefig("../images/countries_global_share_individual.png")

# Subfield probabilities

In [ ]:
year = 2023

In [ ]:
df_country_subfield_by_types_list = []
df_probabilities_by_types_list = []
for type_loop in type_list:
    df_country_subfield_tmp = read_country_subfield(year, type_loop, df_global)
    df_country_subfield_by_types_list.append(
        get_flat_country_subfield(df_country_subfield_tmp).assign(type_name=type_loop)
    )
    df_probabilities_by_types_list.append(
        get_flat_country_subfield(df_country_subfield_tmp.div(df_country_subfield_tmp.sum(axis=1), axis=0))
        .assign(type_name=type_loop)
    )

In [ ]:
df_country_subfield_tmp.div(df_country_subfield_tmp.sum(axis=1), axis=0)

In [ ]:
df_probabilities_by_types = (
    pd.concat(df_probabilities_by_types_list)
    .assign(subfield=lambda df: df.subfield_id.astype(int))
    .merge(uf.df_topics[["subfield_id", "subfield_name", "field_name", "domain_name"]].drop_duplicates(),
           how="left", left_on="subfield", right_on="subfield_id")
)

In [ ]:
df_counbtry_subfield_tmp = read_country_subfield(year, type_loop, df_global)

In [ ]:
df_probabilities_by_types

In [ ]:
year = 2023

fig = px.box(
    df_probabilities_by_types,
    x="domain_name",           # each domain = one box
    y="counts",       # values to summarize
    points="outliers",              # optional: show all points
    color="type_name",       # color boxes by domain
    height=600,
    # title=f"Distribution of counts_loss_pct by domain in {year}",
    hover_data=["domain_name", "counts", "subfield_name", "country"],
)

# fig.update_layout(
#     yaxis_title="Counts loss (%)",
#     xaxis_title="Domain",
#     showlegend=False
# )

fig.show()
# fig.write_image("../images/global_collaboration_losses_by_domain_2023.png",
#                 width=1200, height=600, scale=2)